# Profiling kernels with Nsight Systems and Nsight Compute

A hands-on workflow for finding out why a kernel or a model step is slow: Nsight Systems for the whole timeline (gaps, launches, copies, NVTX ranges), Nsight Compute for one kernel (speed of light, memory chart, occupancy, warp stall reasons, roofline), the PyTorch profiler when you can't use either, and how to turn each finding into a specific fix.

This notebook holds the code from the lesson **[Profiling kernels with Nsight Systems and Nsight Compute](https://inferenceacademy.com/learn/kern-profiling/profiling-with-nsight-systems-and-nsight-compute)** on Inference Academy, in the same order. Read the lesson for the explanations, diagrams and exercises, and run the cells here as you go.

**Before you start:** choose *Runtime → Change runtime type → T4 GPU*, then run the first cell to check the GPU.

In [ ]:
!nvidia-smi

## Two tools, two questions

## Nsight Systems: the timeline

The next cell uses `decode_step.py`, which you put together yourself from the lesson's snippets (or an earlier lesson). Create it first in a cell that starts with `%%writefile decode_step.py`.

In [ ]:
!nsys profile -o step --trace=cuda,nvtx,osrt --force-overwrite true python decode_step.py
!nsys stats step.nsys-rep --report cuda_gpu_kern_sum

### Add NVTX ranges

Illustration from the lesson (not meant to run on its own):

```python
import torch
with torch.cuda.nvtx.range("decode_step"):
    for i, layer in enumerate(model.layers):
        with torch.cuda.nvtx.range(f"layer_{i}"):
            with torch.cuda.nvtx.range("attention"):
                h = layer.attn(x)
            with torch.cuda.nvtx.range("mlp"):
                x = layer.mlp(h)
```

### What to look for

## Nsight Compute: one kernel at a time

The next cell uses `bench_rmsnorm.py`, which you put together yourself from the lesson's snippets (or an earlier lesson). Create it first in a cell that starts with `%%writefile bench_rmsnorm.py`.

In [ ]:
# The first 3 launches of kernels whose names match "rmsnorm", with the full metric set
!ncu --set full -k regex:rmsnorm -c 3 -o rms python bench_rmsnorm.py
# Or print a quick summary to the console
!ncu --section SpeedOfLight -k regex:rmsnorm -c 1 python bench_rmsnorm.py

### Read it in this order

## Profiling the kernels from this course

## When you can't use Nsight: the PyTorch profiler

Illustration from the lesson (not meant to run on its own):

```python
from torch.profiler import profile, ProfilerActivity
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True) as prof:
    for _ in range(5): model(x)
    torch.cuda.synchronize()
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=15))
prof.export_chrome_trace("trace.json")   # open in ui.perfetto.dev
```

## A profiling workflow you can repeat

## What you can now do

## Common mistakes

## Exercise

## Going deeper

---
Code from [Inference Academy](https://inferenceacademy.com), MIT licensed. Lesson: https://inferenceacademy.com/learn/kern-profiling/profiling-with-nsight-systems-and-nsight-compute